# 01 — Build & Verify `PageMantle-Ext` (64 DB-Grounded Tasks)

Constructs leak-free enterprise splits grounded strictly in PageMantle's local 11-service databases:
- `pm_official_8.jsonl` (8 official Deepnote public tasks — **strictly held out**)
- `pm_online_stream_24.jsonl` (18 Action + 6 Silence Traps across 3 waves of 8 — **online DB-CSE stream**)
- `pm_eval_32.jsonl` (22 Action + 10 Hard-Negative Silence Traps — **strictly held out**)
- `verification_audit.json` (Deterministic DB-existence check + Terra ambiguity audit + split isolation)

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"  # Edit once with your repo URL
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    !pip install -q -e . --no-deps

import importlib, sys, types

# Colab Python 3.12+/3.13 removed stdlib `imp`, which older IPython autoreload still imports.
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


In [ ]:
# === CELL 2: Bootstrap Secrets & Resolved Model Config (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import describe_secret_sources
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.config import load_models_config, load_dial_skill_config, load_experiments_config

print("Secret lookup diagnostics:", json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT), indent=2))
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print("Resolved models:", RESOLVED_MODELS.model_dump() if RESOLVED_MODELS else "Not yet generated (run Notebook 00)")


## Step 1: Inspect 11-Service PageMantle Disk-Backed Harness

In [ ]:
from dial_skill.benchmarks.pagemantle_env import ALL_11_SERVICES, PageMantleEnv

BENCH_REPO = REPO_DIR / "external" / "agent-initiative-benchmark"
env = PageMantleEnv(
    dataset_dir=DRIVE_ROOT / "datasets" / "pagemantle",
    benchmark_repo_dir=BENCH_REPO if BENCH_REPO.exists() else None,
)
print("Available services on disk:", env.available_services())
sample_facts = env.mine_verified_facts(limit_per_service=3)
print(f"Sampled {len(sample_facts)} verified DB records across services.")


## Step 2: Build 64-Task Suite & Run Two-Stage Quality Assurance Gate

In [ ]:
from dial_skill.benchmarks.build_pagemantle_ext import build_pagemantle_ext_suite
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

DRY_RUN = True  # Flip to False to enable Terra dialogue realism enrichment & ambiguity audit (~$1.50)
OUT_DIR = DRIVE_ROOT / "datasets" / "pagemantle_ext"

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai = None if DRY_RUN else OpenAIClient(models_cfg=MODELS_CFG, cache=cache)
terra_model = resolve_model_for_role("judge", MODELS_CFG, RESOLVED_MODELS)

audit = await build_pagemantle_ext_suite(
    env=env,
    output_dir=OUT_DIR,
    benchmark_repo_dir=BENCH_REPO if BENCH_REPO.exists() else None,
    openai_client=oai,
    synth_model=None if DRY_RUN else terra_model,
    judge_model=None if DRY_RUN else terra_model,
    seed=EXP_CFG["seed"],
)
print("Verification Audit Report:")
print(json.dumps(audit, indent=2))


## Step 3: Sanity Inspect Split Distribution & Zero-Leakage Assertion

In [ ]:
from dial_skill.benchmarks.build_pagemantle_ext import verify_split_isolation
from dial_skill.schemas import PageMantleTask

def _load_split(name: str):
    p = OUT_DIR / f"{name}.jsonl"
    return [PageMantleTask.model_validate_json(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

official = _load_split("pm_official_8")
stream = _load_split("pm_online_stream_24")
eval_set = _load_split("pm_eval_32")

iso = verify_split_isolation(official, stream, eval_set, strict_counts=True)
print("Split isolation verified (Invariant #8):", json.dumps(iso, indent=2))
